# Buổi 4 — Hồi quy & PCA: các chỉ số trường học có 'giải thích' điểm thô không?

In [1]:
# Chạy ô này đầu tiên. Dữ liệu (PISA 2025, Việt Nam) được tải trực tiếp từ GitHub ở ô kế tiếp — không cần tải/upload tay.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf, statsmodels.api as sm
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",3)

In [2]:
# ---- TẢI DỮ LIỆU (2 bảng: cấp trường 195 dòng, cấp học sinh 7.368 dòng) ----
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
hs = pd.read_csv(RAW + "vnm_hocsinh_7368.csv")
VUNG={1:"ĐB sông Cửu Long",2:"Bắc TB, DH TB & Tây Nguyên",3:"Trung du & MN phía Bắc",4:"ĐB sông Hồng",5:"Đông Nam Bộ"}
truong["vung_ten"]=truong.vung.map(VUNG); truong["loai"]=truong.PRIVATESCH.map({1:"Công",2:"Tư"})
W=["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP","AVLRSOFT","ENCOURPG"]
print("Bảng trường:",truong.shape,"| Bảng học sinh:",hs.shape)

Bảng trường: (195, 36) | Bảng học sinh: (7368, 54)


In [3]:
truong["tu"]=(truong.PRIVATESCH==2).astype(int)
for i in [2,3,4,5]: truong[f"v{i}"]=(truong.vung==i).astype(int)
f1="sci_mean~EDUSHORT+STAFFSHORT+NEGSCLIM+DIGPREP+AVLRSOFT+EDULEAD"
a1=smf.ols(f1,truong).fit(); a2=smf.ols(f1+"+tu+v2+v3+v4+v5",truong).fit()
print(f"Bước 1 (chỉ số trường): R²={a1.rsquared:.3f}, R² hc={a1.rsquared_adj:.3f}, p(F)={a1.f_pvalue:.2f}")
print(f"Bước 2 (+ loại trường + vùng): R²={a2.rsquared:.3f}, R² hc={a2.rsquared_adj:.3f}, p(F)={a2.f_pvalue:.1g}")
print(a2.summary().tables[1])

Bước 1 (chỉ số trường): R²=0.033, R² hc=0.002, p(F)=0.39
Bước 2 (+ loại trường + vùng): R²=0.187, R² hc=0.138, p(F)=6e-05
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4439      0.014     30.916      0.000       0.416       0.472
EDUSHORT      -0.0010      0.007     -0.143      0.887      -0.014       0.012
STAFFSHORT    -0.0010      0.008     -0.129      0.898      -0.016       0.014
NEGSCLIM       0.0063      0.005      1.252      0.212      -0.004       0.016
DIGPREP       -0.0020      0.005     -0.391      0.696      -0.012       0.008
AVLRSOFT       0.0107      0.007      1.612      0.109      -0.002       0.024
EDULEAD       -0.0001      0.007     -0.017      0.986      -0.014       0.014
tu            -0.0275      0.017     -1.658      0.099      -0.060       0.005
v2            -0.0139      0.019     -0.745      0.457      -0.051       0.023
v3       

**❓** R² tăng từ 0.03 lên 0.19 — phần tăng đến từ biến nào? Kết luận nào đúng: 'chỉ số trường học không ảnh hưởng điểm' hay 'trong dữ liệu này không thấy liên hệ'? Vì sao khác nhau?

## Đa cộng tuyến (PSPP không có VIF)

In [4]:
r=truong.EDUSHORT.corr(truong.STAFFSHORT); print("r(EDUSHORT,STAFFSHORT)=",round(r,2),"→ VIF ≈",round(1/(1-r**2),2))
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
X=truong[W].assign(c=1); print({c:round(vif(X.values,i),2) for i,c in enumerate(X.columns) if c!="c"})

r(EDUSHORT,STAFFSHORT)= 0.61 → VIF ≈ 1.61
{'EDULEAD': np.float64(1.32), 'NEGSCLIM': np.float64(1.11), 'STAFFSHORT': np.float64(1.72), 'EDUSHORT': np.float64(1.65), 'DIGPREP': np.float64(1.13), 'AVLRSOFT': np.float64(1.08), 'ENCOURPG': np.float64(1.29)}


## PCA trên 7 chỉ số (giảm chiều — không phải 'tìm nhân tố thật')

In [5]:
Z=truong[W]; R=np.corrcoef(Z.T.values); w,V=np.linalg.eigh(R); idx=np.argsort(w)[::-1]; w,V=w[idx],V[:,idx]
n,p=Z.shape; chi=-(n-1-(2*p+5)/6)*np.log(np.linalg.det(R)); inv=np.linalg.inv(R); pc=-inv/np.sqrt(np.outer(np.diag(inv),np.diag(inv))); off=~np.eye(p,dtype=bool)
print("Eigenvalue:",w.round(2)); print(f"Bartlett χ²={chi:.1f} (df={p*(p-1)//2}); KMO={(R[off]**2).sum()/((R[off]**2).sum()+(pc[off]**2).sum()):.2f}")
def varimax(L,it=100,tol=1e-8):
    p,k=L.shape; Rm=np.eye(k); d=0
    for _ in range(it):
        Lr=L@Rm; u,s,vt=np.linalg.svd(L.T@(Lr**3-Lr@np.diag((Lr**2).sum(0))/p)); Rm=u@vt
        if s.sum()<d*(1+tol): break
        d=s.sum()
    return L@Rm
k=int((w>1).sum()); print(pd.DataFrame(varimax(V[:,:k]*np.sqrt(w[:k])),index=W,columns=[f"PC{i+1}" for i in range(k)]).round(2))

Eigenvalue: [2.09 1.41 1.01 0.82 0.77 0.53 0.37]
Bartlett χ²=191.4 (df=21); KMO=0.61
             PC1   PC2   PC3
EDULEAD    -0.04  0.81 -0.15
NEGSCLIM    0.52 -0.22 -0.48
STAFFSHORT  0.87 -0.01  0.01
EDUSHORT    0.82 -0.01  0.15
DIGPREP    -0.37  0.25 -0.45
AVLRSOFT   -0.10  0.14 -0.81
ENCOURPG   -0.02  0.86 -0.01


## Cấp học sinh: điểm thô Khoa học ~ Toán / Đọc / LDW (những em có cả hai)

In [6]:
for x in ["math_mc","read_mc","ldw_mc"]:
    d=hs.dropna(subset=["sci_mc",x]); m=smf.ols(f"sci_mc~{x}",d).fit(); print(f"{x}: n={len(d)}, B={m.params[x]:.3f}, R²={m.rsquared:.3f}")

math_mc: n=2258, B=0.420, R²=0.377
read_mc: n=2280, B=0.528, R²=0.258
ldw_mc: n=1479, B=0.182, R²=0.315


## Bài tập
`bai_tap/buoi4_de.md`